# 07 • PDF RAG: คู่มืองานคลังและการส่งมอบ

เป้าหมาย: อัปโหลด PDF ภาษาไทย แยกข้อความ เก็บเลขหน้า สร้าง Embedding และตอบพร้อมอ้างอิงหน้า

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[เปิด notebook นี้ใน Colab](https://colab.research.google.com/drive/18zSRK-AsaoEn961nRzetXxhHZyWdJmuF?usp=drive_link)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai numpy pymupdf

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


## อัปโหลดเอกสารและอ่านข้อความ
ใช้ sritrang_training_sop_th.pdf ตัวอย่างสมมติ อัปโหลดผ่าน Files pane ทางซ้ายก่อนรันได้ หรือใช้ปุ่มเลือกไฟล์ใน output หากไม่มีไฟล์ใน runtime ต้องใช้ PDF ที่มี text layer


In [ ]:
from google.colab import files
import pymupdf

uploaded = files.upload()
pdf_name = next(iter(uploaded))
doc = pymupdf.open(stream=uploaded[pdf_name], filetype="pdf")
chunks = []
for page_number, page in enumerate(doc, 1):
    text = page.get_text()
    for start in range(0, len(text), 600):
        chunks.append(f"[หน้า {page_number}] " + text[start:start + 750])
if not chunks:
    raise ValueError("ไม่พบข้อความ กรุณาใช้ PDF ที่มี text layer")
print("จำนวนหน้า:", len(doc), "จำนวน chunks:", len(chunks))
print(chunks[0])

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

In [ ]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

In [ ]:
question = "เมื่อ Lot ยังไม่มีผล QC ฝ่ายคลังและฝ่ายขนส่งต้องทำอะไรตามคู่มือนี้?"
hits = retrieve(question, k=3)
context = "\n".join(text for text, score in hits)
print(context)
print(client.responses.create(
    model=MODEL,
    instructions="ตอบเป็นภาษาไทยโดยใช้เฉพาะข้อความที่ให้ อ้างเลขหน้า หากข้อมูลไม่มีให้ระบุว่าไม่พบในเอกสาร ห้ามอนุมานขั้นตอนเพิ่ม หากเอกสารไม่ได้ระบุหน้าที่ฝ่ายขนส่งโดยตรง ให้บอกว่าไม่ได้ระบุหน้าที่ฝ่ายขนส่งโดยตรง",
    input=f"ข้อความที่ค้นพบ:\n{context}\n\nคำถาม: {question}",
    max_output_tokens=500,
).output_text)

## ลองทำด้วยตนเอง
1. ถามขั้นตอนจัดการ Lot ที่ยังไม่มีผล QC
2. ถามเรื่องการเลื่อนรถรับสินค้าระหว่างน้ำท่วม แล้วตรวจเลขหน้าใน PDF
3. ถามเรื่องที่ไม่มีในคู่มือ ตรวจ chunks ที่ได้ก่อนประเมินคำตอบ
